In [1]:
import chromadb
from sentence_transformers import SentenceTransformer
import pickle
import re

/home/kxelina/RAG_project/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Initialize embedding model 
embedding_model = SentenceTransformer('BAAI/bge-small-en-v1.5')
embedding_dim = embedding_model.get_sentence_embedding_dimension()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7096.36it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
client = chromadb.PersistentClient(path="./chroma_db")
for col in ["global_summaries", "grouped_summaries", "raw_transactions"]:
    try: client.delete_collection(name=col)
    except: pass

collection_global = client.get_or_create_collection(name="global_summaries", metadata={"hnsw:space": "cosine"})
collection_grouped = client.get_or_create_collection(name="grouped_summaries", metadata={"hnsw:space": "cosine"})
collection_raw = client.get_or_create_collection(name="raw_transactions", metadata={"hnsw:space": "cosine"})

In [4]:
# Load preprocessed data
with open("analysis.pkl", "rb") as f:
    data = pickle.load(f)

global_docs = data['global']['documents']
grouped_docs = data['grouped']['documents']
raw_docs = data['raw']['documents']

print(f"  Global: {len(global_docs)} chunks")
print(f"  Grouped: {len(grouped_docs)} chunks")
print(f"  Raw: {len(raw_docs)} chunks")

  Global: 54 chunks
  Grouped: 1312 chunks
  Raw: 9994 chunks


In [5]:
# Helper function to add documents to collection
def add_to_collection(collection, documents, collection_name, batch_size=50):
    total = len(documents)
    
    for i in range(0, total, batch_size):
        batch = documents[i:i + batch_size]
        
        ids = [f"{collection_name}_{doc.metadata['chunk_id']}" for doc in batch]
        texts = [doc.page_content for doc in batch]
        metadatas = [doc.metadata for doc in batch]
        
        embeddings = embedding_model.encode(texts, show_progress_bar=False).tolist()
        
        collection.add(
            ids=ids,
            embeddings=embeddings,
            documents=texts,
            metadatas=metadatas
        )

In [6]:
# Add layers to their respective collections
add_to_collection(collection_global, global_docs, "global")
add_to_collection(collection_grouped, grouped_docs, "grouped")
add_to_collection(collection_raw, raw_docs, "raw")

In [16]:
def extract_amount(text):
    match = re.search(r'\$[\d,]+\.?\d*', text)
    return float(match.group().replace('$', '').replace(',', '')) if match else 0

def extract_margin(text):
    match = re.search(r'Margin\s+([-\d.]+)%', text)
    return float(match.group(1)) if match else 0

def extract_discount(text):
    match = re.search(r'Average discount (\d+)%', text)
    return float(match.group(1)) if match else 0

def retrieve_context(query, num_results=5, fact_type=None):
    if fact_type == "subcategory" or fact_type == "monthly":
        retrieve_size = 100  
    elif "discount" in query.lower() or "frequently sold" in query.lower():
        retrieve_size = 2000
    else:
        retrieve_size = 25 if fact_type else num_results

    if "discount" in query.lower() or "frequently sold" in query.lower():
        grouped_results = collection_grouped.query(
            query_texts=["frequently discounted"],  
            n_results=2000,  
            include=["documents", "metadatas"])
        
        documents = []
        if grouped_results['documents'] and len(grouped_results['documents']) > 0:
            docs_list = grouped_results['documents'][0]
            
            for doc in docs_list:
                if "Frequently discounted product" in doc:
                    documents.append(doc)
        
        # Sort by discount percentage (descending)
        documents = sorted(documents, key=extract_discount, reverse=True)
        documents = documents[:num_results]
        
        context = "\n".join(documents)
        if len(context) > 2500:
            context = context[:2500]
        
        return context, len(documents)
    
    # similarity search on both collections
    global_results = collection_global.query(
        query_texts=[query],
        n_results=retrieve_size,
        include=["documents", "metadatas"]
    )

    grouped_results = collection_grouped.query(
        query_texts=[query], 
        n_results=retrieve_size,
        include=["documents", "metadatas"])
    
    # metadata filtering and aggregation
    documents = []
    if global_results['documents'] and len(global_results['documents']) > 0:
        docs_list = global_results['documents'][0]
        meta_list = global_results['metadatas'][0] if global_results['metadatas'] else []
        
        for doc, meta in zip(docs_list, meta_list):
            if fact_type is not None and meta.get('fact_type') != fact_type:
                continue
            if not meta.get('is_aggregate', True):
                continue
            documents.append(doc)
    
    if grouped_results['documents'] and len(grouped_results['documents']) > 0:
        docs_list = grouped_results['documents'][0]
        meta_list = grouped_results['metadatas'][0] if grouped_results['metadatas'] else []
        
        for doc, meta in zip(docs_list, meta_list):
            if fact_type is not None and meta.get('fact_type') != fact_type:
                continue    
            documents.append(doc)
    
    # Sort by business metric
    if "margin" in query.lower():
        documents = sorted(documents, key=extract_margin, reverse=True)
    else:
        documents = sorted(documents, key=extract_amount, reverse=True)
    
    # Take top N
    documents = documents[:num_results]
    
    context = "\n".join(documents)
    if len(context) > 2500:
        context = context[:2500]
    
    return context, len(documents)

In [17]:
# classify query for filtering
def classify_query_for_filter(query):
    q = query.lower()
    
    if "compare" in q and ("category" in q or "technology" in q or "furniture" in q or "office" in q):
        return "category", 5
    elif "technology" in q and "furniture" in q:
        return "category", 5

    if "trend" in q or "4-year" in q or "over time" in q or "changed over" in q:
        return "yearly", 5
    elif "month" in q or "seasonal" in q or "november" in q or "december" in q:
        return "monthly", 15
    elif "category" in q or "product category" in q:
        return "category", 5
    elif "sub-categor" in q or "sub-cat" in q or "margin" in q:  
        return "subcategory", 8
    elif "region" in q or "west" in q or "east" in q or "central" in q or "south" in q:
        return "region", 5
    elif "state" in q or "california" in q or "new york" in q or "texas" in q:
        return "state", 10
    elif "city" in q or "cities" in q or "top performer" in q:  
        return "city", 10
    elif "discount" in q or "frequently sold" in q:  
        return None, 10
    else:
        return None, 5

In [19]:
# test queries with expected answers for validation
test_queries = [
    ("What is the sales trend over the 4-year period?", 
     "EXPECTED: 2014: $484,247.50, 2015: $470,532.51, 2016: $609,205.60, 2017: $733,215.26 (51.5% growth)"),
    
    ("Which months show the highest sales? Is there seasonality?", 
     "EXPECTED: November ($352,461.07), December ($325,293.50), September ($307,649.95) - Peak in Nov-Dec"),
    
    ("How has profit margin changed over time?", 
     "EXPECTED: 2014: 11.81%, 2015: 11.76%, 2016: 12.98%, 2017: 11.60%"),
    
    ("Which product category generates the most revenue?", 
     "EXPECTED: Technology ($836,154.03, 36.4%), Furniture ($741,999.80, 32.3%), Office Supplies ($719,047.03, 31.3%)"),
    
    ("What sub-categories have the highest profit margins?", 
     "EXPECTED: Labels (44.42%), Paper (43.39%), Envelopes (42.27%), Copiers (37.20%), Fasteners (31.40%)"),
    
    ("Which products are frequently sold at a discount?", 
     "EXPECTED: Acco 6 Outlet Guardian Premium Plus Surge Suppressor (80%), Belkin F9S820V06 8 Outlet Surge (80%), Acco 6 Outlet Guardian Basic Surge Suppressor (80%)"),
    
    ("Which region has the best sales performance?", 
     "EXPECTED: West ($725,457.82), East ($678,781.24), Central ($501,239.89), South ($391,721.91)"),
    
    ("Compare sales performance across different states.", 
     "EXPECTED: California ($457,687.63), New York ($310,876.27), Texas ($170,188.05), Washington ($138,641.27), Pennsylvania ($116,511.91)"),
    
    ("Which cities are the top performers?", 
     "EXPECTED: New York City ($256,368.16), Los Angeles ($175,851.34), Seattle ($119,540.74), San Francisco ($112,669.09), Philadelphia ($109,077.01)"),
    
    ("Compare Technology vs Furniture sales trends.", 
     "EXPECTED: Technology ($836,154.03) > Furniture ($741,999.80), Technology leading"),
    
    ("How does the West region compare to the East in terms of profit?", 
     "EXPECTED: West ($108,418.45, 14.94% margin) > East ($91,522.78, 13.48% margin)")
]

for i, (query, expected) in enumerate(test_queries, 1):
    print(f"\n{i}. {query}")
    print(f"   {expected}")
    print("-" * 70)
    fact_type, num_results = classify_query_for_filter(query)
    context, num_docs = retrieve_context(query, fact_type=fact_type, num_results=num_results)
    print(context)
    print()


1. What is the sales trend over the 4-year period?
   EXPECTED: 2014: $484,247.50, 2015: $470,532.51, 2016: $609,205.60, 2017: $733,215.26 (51.5% growth)
----------------------------------------------------------------------
Year 2017 TOTAL: Sales $733,215.26, Profit $93,439.27, Margin 11.60%
Year 2016 TOTAL: Sales $609,205.60, Profit $81,795.17, Margin 12.98%
Year 2014 TOTAL: Sales $484,247.50, Profit $49,543.97, Margin 11.81%
Year 2015 TOTAL: Sales $470,532.51, Profit $61,618.60, Margin 11.76%


2. Which months show the highest sales? Is there seasonality?
   EXPECTED: November ($352,461.07), December ($325,293.50), September ($307,649.95) - Peak in Nov-Dec
----------------------------------------------------------------------
Peak sales month November: $352,461.07 with high seasonality
Peak sales month December: $325,293.50 with high seasonality
Peak sales month September: $307,649.95 with high seasonality
Peak sales month March: $205,005.49 with high seasonality
Peak sales month O

In [11]:
total = collection_global.count() + collection_grouped.count() + collection_raw.count()
print(f"Vector DB: {total} total chunks")

Vector DB: 11360 total chunks
